In [0]:
# Databricks notebook source
# Chunk documents from bronze → silver (OPTIMIZED)

import uuid
from pyspark.sql.functions import col
from datetime import datetime

CATALOG = "vb_rag_demo"   
SCHEMA = "rag_demo"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_documents"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_chunks"

CHUNK_SIZE = 800          
CHUNK_OVERLAP = 100       

# -----------------------------
# CHUNK FUNCTION
# -----------------------------
def chunk_text(text: str, chunk_size=800, overlap=100):
    text = " ".join((text or "").split())
    start = 0
    chunks = []

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])

        if end >= len(text):
            break

        start = end - overlap

    return chunks

# -----------------------------
# READ DATA (NO COLLECT)
# -----------------------------
df = spark.table(BRONZE_TABLE).select(
    "document_id", "file_name", "file_path", "raw_text"
)

# -----------------------------
# PROCESS ROWS SAFELY
# -----------------------------
rows = []

for row in df.toLocalIterator():   #  Better than collect()
    pieces = chunk_text(row["raw_text"], CHUNK_SIZE, CHUNK_OVERLAP)

    for i, chunk in enumerate(pieces):
        rows.append((
            str(uuid.uuid4()),
            row["document_id"],
            row["file_name"],
            chunk,
            i,
            row["file_path"],
            datetime.utcnow()
        ))

# -----------------------------
# WRITE TO SILVER TABLE
# -----------------------------
if rows:
    chunk_df = spark.createDataFrame(rows, [
        "chunk_id",
        "document_id",
        "file_name",
        "chunk_text",
        "chunk_order",
        "source_path",
        "ingested_at"
    ])

    chunk_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER_TABLE)

    print(f" Created {len(rows)} chunks in {SILVER_TABLE}")
    print(f"Chunk size: {CHUNK_SIZE}, Overlap: {CHUNK_OVERLAP}")
else:
    print(" No chunks created")

In [0]:
%sql
SELECT COUNT(*) FROM vb_rag_demo.rag_demo.silver_chunks;

In [0]:
%sql
SELECT file_name, COUNT(*) as chunks
FROM vb_rag_demo.rag_demo.silver_chunks
GROUP BY file_name;